# DATASET DE MARLIM POR REGIÃO
Gera `Dataset/marlim_nature` com `PER_REGION` tiles de cada uma das três regiões — `calm`, `faulted`
e `dead` — usando a configuração que a busca do `Synthetic/Analysis.ipynb` gravou em
`Synthetic/regions.json`, uma por região.

O tile sai do `SyntheticGenerator` z-scorado, leva o **ganho da região** (o escalar que põe o σ dele
no σ mediano dos tiles reais daquela região) e o **jitter** do tile, e é gravado **cru**, saturado em
±`CLIP`, em `original/images`. O ganho é por região justamente para preservar o contraste relativo do
bloco — a zona morta real é ~12× mais fraca que o pacote raso —, que um z-score por tile apagaria. O
jitter usa o mesmo sorteio normalizado do `Synthetic/Analysis.ipynb` (mediana 1), que é onde o ganho
foi ajustado; sem normalizar, o σ do dataset dependeria de quantos tiles ele tem.

**O `Format` desta pasta declara os trilhos em vez de medi-los.** Nos outros datasets o `p01`/`p99`
sai do percentil da amostra, e lá isso dá exatamente ±`CLIP` porque o gerador do `Generator2.ipynb`
satura mais de 1% dos voxels. Aqui não: com o ganho de cada região, o `calm` encosta em ~2% dos
voxels, o `faulted` em ~0.03% e o `dead` em nenhum, então o percentil da amostra cairia dentro dos
trilhos **e dependeria de quantos tiles cada região tem**. A escala [0,1] precisa ser exatamente
`(clip(img, ±CLIP) + CLIP) / 2·CLIP`, que é aquela em que a similaridade foi medida e em que o bloco
real vive, então `export` grava uma cópia do `Format.ipynb` com essa única célula trocada: os
trilhos vêm do `clipLevel` do `synthetic.json`. O resto do notebook é o padrão, sem mudança.

| passo | onde |
|---|---|
| tiles crus + `synthetic.json` + `Format.ipynb` | este notebook |
| `images/`, `masks/`, `DataBase.csv` | `Dataset/marlim_nature/Format.ipynb` |


In [ ]:
import os, sys, json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime
from tqdm import tqdm

sys.path.append('../..')
from Synthetic.index import SyntheticGenerator
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [ ]:
DATASET    = '../../Dataset/marlim_nature'
FORMAT_SRC = '../../Dataset/dataset_marlim_opt_2/Format.ipynb'
CONFIG     = 'Synthetic/regions.json'
REAL_DB    = 'Marlim/files/DataBase.csv'

TILE        = 128
PER_REGION  = 220        # 3 regiões -> 660 tiles, o mesmo tamanho do dataset_marlim_opt_2
CLIP        = 0.45       # saturação simétrica do slab real; é o trilho que o Format desta pasta declara
SEED        = 20260912   # semente do tile = SEED + 1000 * índice da região + índice do tile
SIGMA_TOL   = 0.10       # desvio aceito entre o σ do tile gerado e o σ mediano dos tiles reais da região
NJOBS       = min(18, os.cpu_count())

PER_REGION, PER_REGION * 3, CLIP

# CONFIGURAÇÃO DAS REGIÕES
Cada região traz da busca as opções do gerador, o ganho ajustado ao σ real e a nota que ela alcançou.


In [ ]:
config = json.load(open(CONFIG))
real   = pd.read_csv(REAL_DB)

REGIONS = {name: dict(cfg, sigma=float(real[real.region == name].img_std.median()))
           for name, cfg in config['regions'].items()}

print(f"regions.json atualizado em {config['updated']}  |  nota medida em {config['nImages']} tiles por lote")

pd.DataFrame({name: {'nota': cfg['score'], 'origem': cfg['source'], 'ganho': round(cfg['gain'], 5),
                     'jitter': cfg['genome']['gainJitter'], 'σ real': round(cfg['sigma'], 4),
                     'tiles': PER_REGION, **cfg['groups']}
              for name, cfg in REGIONS.items()}).T

# GERAÇÃO
`RegionDataset` apaga a pasta, gera as três regiões em paralelo e grava `synthetic.json` com tudo que
reproduz o dataset — ganho, jitter, semente e as opções de cada região. A semente de cada tile é
`SEED + 1000 × índice da região + índice do tile`, então nenhuma região repete tile de outra e o
dataset inteiro é reproduzível.


In [ ]:
class RegionDataset:
    RAILS = """p99 = json.load(open('synthetic.json'))['clipLevel']
p01 = -p99
print('trilhos declarados:', p01, p99)

"""

    def __init__(self, path, regions, n=PER_REGION, seed=SEED, clip=CLIP):
        self.path    = path
        self.regions = regions
        self.n       = n
        self.seed    = seed
        self.clip    = clip
        self.images  = f'{path}/original/images'
        self.masks   = f'{path}/original/masks'

    # GERA AS TRÊS REGIÕES EM original/ E DEIXA O synthetic.json E O Format.ipynb DO LADO
    def update(self):
        shutil.rmtree(self.path, ignore_errors=True)
        os.makedirs(self.images)
        os.makedirs(self.masks)
        self.stats = pd.concat([self.process(name, i) for i, name in enumerate(self.regions)], ignore_index=True)
        return self.export()

    # UMA REGIÃO: TILES CRUS COM O GANHO DELA E O JITTER DO TILE, SATURADOS EM ±clip
    def process(self, region, index):
        cfg    = self.regions[region]
        params = {k: tuple(v) if isinstance(v, list) else v for k, v in cfg['params'].items()}
        seeds  = [self.seed + 1000 * index + i for i in range(self.n)]
        jitter = self.jitter(cfg['genome']['gainJitter'], self.seed + index, self.n)
        tasks  = [(f'{region}_{i:03d}', params, s, cfg['gain'] * j, self.clip, self.images, self.masks)
                  for i, (s, j) in enumerate(zip(seeds, jitter))]

        with ProcessPoolExecutor(NJOBS, mp_context=mp.get_context('fork')) as pool:
            rows = list(tqdm(pool.map(self.tile, tasks), total=self.n, desc=region))

        return pd.DataFrame(rows).assign(region=region)

    # O JITTER DE CONTRASTE DO LOTE, COM MEDIANA 1 — O MESMO SORTEIO DO Synthetic/Analysis.ipynb, ONDE
    # O GANHO FOI AJUSTADO; SEM NORMALIZAR, O σ DO DATASET DEPENDERIA DE QUANTOS TILES ELE TEM
    @staticmethod
    def jitter(amount, seed, n):
        draw = np.exp(amount * np.clip(np.random.RandomState(seed).normal(0, 1, n), -2, 2))
        return draw / np.median(draw)

    # UM TILE GRAVADO: (inline, z, xline) COMO O SLAB REAL, IMAGEM float32 CRUA E MÁSCARA uint8
    @staticmethod
    def tile(args):
        name, params, seed, gain, clip, images, masks = args
        np.random.seed(seed)
        gen = SyntheticGenerator(shape=(TILE,) * 3)
        gen.set(params)
        img, msk = gen.get()
        img = np.clip(np.transpose(img, (0, 2, 1)) * gain, -clip, clip).astype(np.float32)
        msk = np.transpose(msk, (0, 2, 1)).astype(np.uint8)

        np.save(f'{images}/{name}.npy', img)
        np.save(f'{masks}/{name}.npy', msk)
        return {'id': f'{name}.npy', 'seed': seed, 'gain': gain, 'img_std': float(img.std()),
                'img_min': float(img.min()), 'img_max': float(img.max()),
                'satFrac': float((np.abs(img) >= clip).mean()), 'maskFrac': float(msk.mean())}

    # synthetic.json COM TUDO QUE REPRODUZ O DATASET, MAIS O Format.ipynb COM OS TRILHOS DECLARADOS
    def export(self):
        self.format()
        meta = {'dataset': os.path.basename(self.path),
                'generator': 'Synthetic/index.py :: SyntheticGenerator (padrões da classe)',
                'notebook': 'Marlim/Regions/Generate.ipynb',
                'config': CONFIG, 'shape': [TILE] * 3, 'clipLevel': self.clip,
                'perRegion': self.n, 'total': self.n * len(self.regions), 'seed0': self.seed,
                'seedRule': 'np.random.seed(seed0 + 1000 * índice da região + índice do tile)',
                'axes': '(inline, z, xline), o mesmo do slab real',
                'created': datetime.now().strftime('%d/%m/%y %H:%M:%S'),
                'regions': {name: {'tiles': self.n, 'gain': cfg['gain'], 'jitter': cfg['genome']['gainJitter'],
                                   'sigma': cfg['sigma'], 'score': cfg['score'], 'groups': cfg['groups'],
                                   'params': cfg['params']}
                            for name, cfg in self.regions.items()}}

        json.dump(meta, open(f'{self.path}/synthetic.json', 'w'), ensure_ascii=False, indent=4)
        return self.stats

    # O Format PADRÃO, TROCANDO O PERCENTIL DA AMOSTRA PELO TRILHO DECLARADO EM synthetic.json
    def format(self):
        nb    = json.load(open(FORMAT_SRC))
        cells = [c for c in nb['cells'] if 'np.percentile(arrays, 1)' in ''.join(c['source'])]

        if len(cells) != 1:
            raise ValueError(f'{FORMAT_SRC}: esperava uma célula de percentil, achei {len(cells)}')

        source = ''.join(cells[0]['source'])
        cells[0]['source'] = (self.RAILS + source[source.index('def normalize'):]).splitlines(keepends=True)
        cells[0]['outputs'] = []
        cells[0]['execution_count'] = None
        json.dump(nb, open(f'{self.path}/Format.ipynb', 'w'), ensure_ascii=False, indent=1)

    # O p01/p99 QUE UMA AMOSTRA DE 1 EM CADA 17 VOXELS DARIA, SÓ PARA COMPARAR COM O TRILHO DECLARADO
    def percentiles(self, stride=17):
        sample = np.concatenate([np.load(f'{self.images}/{name}').ravel()[::stride]
                                 for name in sorted(os.listdir(self.images))])
        return float(np.percentile(sample, 1)), float(np.percentile(sample, 99))

    # RELÊ TUDO: CONTAGEM, SHAPE, TIPO, O TRILHO DO Format E O σ DE CADA REGIÃO CONTRA O σ REAL DELA
    def check(self):
        p01, p99 = -self.clip, self.clip
        sample   = self.percentiles()

        if 'np.percentile(arrays' in open(f'{self.path}/Format.ipynb').read():
            raise ValueError(f'{self.path}/Format.ipynb ainda tira o percentil da amostra: a escala '
                             f'sairia dependendo de quantos tiles cada região tem')

        rows = []

        for name in self.regions:
            group = self.stats[self.stats.region == name]
            sigma = []

            for tile in tqdm(group.id, desc=name, leave=False):
                img = np.load(f'{self.images}/{tile}')
                msk = np.load(f'{self.masks}/{tile}')

                if img.shape != (TILE,) * 3 or img.dtype != np.float32 or msk.dtype != np.uint8:
                    raise ValueError(f'{tile}: shape {img.shape}, tipos {img.dtype}/{msk.dtype}')

                if img.min() < -self.clip or img.max() > self.clip or msk.max() > 1:
                    raise ValueError(f'{tile}: faixa [{img.min()}, {img.max()}], máscara até {msk.max()}')

                sigma.append(float(((np.clip(img, p01, p99) - p01) / (p99 - p01)).std()))

            real  = self.regions[name]['sigma']
            ratio = float(np.median(sigma)) / real
            rows.append({'região': name, 'tiles': len(group), 'σ pós-Format': round(float(np.median(sigma)), 4),
                         'σ real': round(real, 4), 'razão': round(ratio, 3),
                         'saturação': round(float(group.satFrac.mean()), 4),
                         'rótulo': round(float(group.maskFrac.mean()), 4),
                         'sem falha': int((group.maskFrac == 0).sum())})

            if abs(ratio - 1) > SIGMA_TOL:
                raise ValueError(f'{name}: σ pós-Format {np.median(sigma):.4f} contra {real:.4f} do real '
                                 f'(razão {ratio:.3f}), fora de ±{SIGMA_TOL:.0%}')

        print(f'trilhos do Format: {p01} / {p99}   (o percentil da amostra daria {sample[0]:.4f} / '
              f'{sample[1]:.4f} — por isso eles são declarados)')
        return pd.DataFrame(rows).set_index('região')

    # A SEÇÃO CENTRAL DE 4 TILES DE CADA REGIÃO, NA ESCALA [0,1] QUE O Format VAI GRAVAR
    def show(self, n=4, seed=3):
        fig, axes = plt.subplots(len(self.regions), n, figsize=(3.6 * n, 3.9 * len(self.regions)))

        for row, name in zip(axes, self.regions):
            picks = self.stats[self.stats.region == name].sample(n, random_state=seed)

            for ax, tile in zip(row, picks.itertuples()):
                img = (np.load(f'{self.images}/{tile.id}')[TILE // 2] + self.clip) / (2 * self.clip)
                msk = np.load(f'{self.masks}/{tile.id}')[TILE // 2]
                zf, xf = np.nonzero(msk)

                ax.imshow(img, cmap='gray', vmin=0, vmax=1)
                ax.scatter(xf, zf, s=0.6, c='red')
                ax.set_title(f'{tile.id}\nσ={img.std():.3f}  falha={msk.mean() * 100:.1f}%', fontsize=9)
                ax.axis('off')

        fig.suptitle(f'{os.path.basename(self.path)} — {len(self.stats)} tiles, seção central na escala do Format', fontsize=13)
        plt.tight_layout(rect=(0, 0, 1, 0.97))
        plt.show()

    def info(self):
        return self.stats.groupby('region').agg(tiles=('id', 'size'), sigma=('img_std', 'median'),
                                                saturacao=('satFrac', 'mean'), rotulo=('maskFrac', 'mean'),
                                                sem_falha=('maskFrac', lambda m: int((m == 0).sum())))


data = RegionDataset(DATASET, REGIONS)
data.update()

data.info().round(4)

# CONFERÊNCIA
`check` relê os 660 arquivos e falha se algo estiver fora do contrato: shape, tipo, faixa, um
`Format.ipynb` que ainda tire percentil da amostra e — o que de fato importa para o treino — o σ de
cada região depois do `Format` contra o σ mediano dos tiles reais daquela região. Ele também imprime
o percentil que a amostra daria, para deixar visível o quanto ele difere do trilho declarado.

A tolerância é `SIGMA_TOL` porque o ganho da busca foi ajustado ao σ das **seções 2D** (a feature
`logStd` da régua, média geométrica de 8 seções), e aqui se mede o σ do **volume inteiro**: no bloco
real as duas medidas diferem 0.5% no `calm`, 1.5% no `dead` e 5.3% no `faulted`, que é a
heterogeneidade entre seções do mesmo tile, não erro de escala.


In [ ]:
data.check()

# AMOSTRA
A seção central de quatro tiles de cada região, já na escala [0,1] que o `Format` vai gravar, com a
máscara de falha em vermelho. Depois disso, rodar `Dataset/marlim_nature/Format.ipynb` fecha o
dataset (`images/`, `masks/`, `DataBase.csv`) como em qualquer outro.


In [ ]:
data.show()